# 06 五比特噪声传播与闭环仿真（6 月）

目标：在不少于五比特系统中刻画噪声传播与累积，并形成闭环仿真样例。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('???????????? pyproject.toml ? examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

In [ ]:
QASM_5Q = """
OPENQASM 3;
qubit[5] q;
bit[5] c;
h q[0];
cx q[0], q[1];
cx q[1], q[2];
cx q[2], q[3];
cx q[3], q[4];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
measure q[3] -> c[3];
measure q[4] -> c[4];
"""

In [ ]:
chain_couplings = [
    {'i':0,'j':1,'g':0.02,'kind':'xx+yy'},
    {'i':1,'j':2,'g':0.02,'kind':'xx+yy'},
    {'i':2,'j':3,'g':0.02,'kind':'xx+yy'},
    {'i':3,'j':4,'g':0.02,'kind':'xx+yy'},
]
noise_scales = [0.5, 1.0, 2.0, 4.0]
mean_excited, total_time = [], []

for s in noise_scales:
    hw = {'qubit_freqs_hz':[5.0e9,5.1e9,4.95e9,5.05e9,5.2e9], 'couplings':chain_couplings}
    nz = {'model':'markovian_lindblad', 'gamma1':0.001*s, 'gamma_phi':0.002*s}
    r = run_case(f'06_scale_{s:.1f}', QASM_5Q, hardware=hw, noise=nz)
    mean_excited.append(get_metric(r, 'mean_excited'))
    total_time.append(float(r['timings'].get('total', np.nan)))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(noise_scales, mean_excited, marker='o')
ax[0].set_title('???? vs ????')
ax[0].set_xlabel('noise scale'); ax[0].set_ylabel('mean_excited'); ax[0].grid(alpha=0.3)

ax[1].plot(noise_scales, total_time, marker='o', color='tab:green')
ax[1].set_title('???? vs ?????')
ax[1].set_xlabel('noise scale'); ax[1].set_ylabel('timings.total (s)'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()